In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import networkx as nx

# Add the parent directory to the path to import from src
sys.path.append(os.path.abspath(".."))
from concept_creator.energy_minimization_concept_service import EnergyMinimizationConceptService
from concept_creator.visualization_helpers import ConceptVisualization

# Configuration 
uri = "bolt://localhost:7687"
user = "neo4j"
password = "111122223333"
session_id = '2_2'
concept_name = session_id

# Initialize the visualization helper
visualizer = ConceptVisualization(figsize=(15, 10))

# Store visualization results
evolution_steps = []
property_diffs = []

# Define visualization callback with property tracking and side-by-side comparison
def concept_visualization_callback(step_number, pre_concept, image_graph, post_concept, mcm_results, concept_to_new):
    print(f"Visualizing step {step_number}...")
    
    # For the first step (initial image)
    if pre_concept is None:
        fig, ax = plt.subplots(figsize=(12, 10))
        visualizer.visualize_graph(image_graph, f"Initial Image (Step {step_number})", ax=ax)
        evolution_steps.append((step_number, fig))
        plt.close(fig)
        return
    
    # For subsequent steps, show both the graph evolution and property changes
    fig = visualizer.visualize_concept_creation_with_props(
        pre_concept, image_graph, post_concept, concept_to_new, step_number)
    
    # Store the result for later viewing
    evolution_steps.append((step_number, fig))
    
    # Store property diffs
    if concept_to_new and mcm_results:
        mcm, concept_to_mcm, image_to_mcm, contractions = mcm_results
        
        # Create property comparison table showing concept vs image vs final
        comparison_fig = visualizer.create_property_comparison_table(
            pre_concept, image_graph, post_concept, concept_to_mcm, image_to_mcm)
        
        # Also create a standard property diff table for more detailed inspection
        prop_diff_fig = visualizer.create_property_diff_table(
            pre_concept, post_concept, concept_to_new)
        
        # Store both
        property_diffs.append((step_number, comparison_fig, prop_diff_fig))
    
    plt.close(fig)

# Create the concept with visualization
service = EnergyMinimizationConceptService(uri, user, password)
concept_id, concept_graph = service.create_concept_incrementally(
    session_id, concept_name, visualization_callback=concept_visualization_callback)

# Interactive widgets for browsing visualizations
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create a better integrated widget for viewing all visualizations per epoch
if evolution_steps:
    # Create available step numbers list from all evolution steps
    available_steps = sorted(list(set([step[0] for step in evolution_steps])))
    
    # Function to view epoch with tabs for different visualizations
    def view_epoch_with_tabs(step_index):
        clear_output(wait=True)
        
        # Find evolution step
        matching_step = None
        for step_number, fig in evolution_steps:
            if step_number == step_index:
                matching_step = (step_number, fig)
                break
        
        # Find property diff
        matching_diff = None
        for step_number, comparison_fig, diff_fig in property_diffs:
            if step_number == step_index:
                matching_diff = (step_number, comparison_fig, diff_fig)
                break
        
        if matching_step:
            step_number, fig = matching_step
            print(f"## Epoch {step_number}")
            
            # Create tab outputs
            evolution_output = widgets.Output()
            with evolution_output:
                display(fig)
            
            tab_children = [evolution_output]
            tab_titles = ['Graph Evolution']
            
            if matching_diff:
                _, comparison_fig, diff_fig = matching_diff
                
                # Create outputs for property visualizations
                comparison_output = widgets.Output()
                diff_output = widgets.Output()
                
                with comparison_output:
                    display(comparison_fig)
                
                with diff_output:
                    display(diff_fig)
                
                tab_children.extend([comparison_output, diff_output])
                tab_titles.extend(['Decision Analysis', 'Property Changes'])
            
            # Create tabs widget
            tabs = widgets.Tab(children=tab_children)
            for i, title in enumerate(tab_titles):
                tabs.set_title(i, title)
            
            display(tabs)
        else:
            print(f"Epoch {step_index} data not found")
    
    # Create dropdown for epoch selection
    epoch_dropdown = widgets.Dropdown(
        options=[(f'Epoch {step}', step) for step in available_steps],
        description='Select Epoch:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )
    
    # Create main output area
    main_output = widgets.Output()
    
    # Link the dropdown to update the output
    def on_epoch_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with main_output:
                clear_output(wait=True)
                view_epoch_with_tabs(change['new'])
    
    epoch_dropdown.observe(on_epoch_change, names='value')
    
    # Initial display
    with main_output:
        if available_steps:
            view_epoch_with_tabs(available_steps[0])
    
    # Display the complete widget with controls
    display(widgets.VBox([epoch_dropdown, main_output]))